In [1]:
# IMPORT LIBRARY
import os
import time
import random
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

In [2]:
import tensorflow as tf
print(tf.__version__)


2.12.0


In [3]:
# PARAMETER GLOBAL & SEED
DATASET_PATH = r"D:\Project Jupyter\Dataset_baru"

train_dir = os.path.join(DATASET_PATH, "train")
val_dir   = os.path.join(DATASET_PATH, "val")
test_dir  = os.path.join(DATASET_PATH, "test")

CLASSES = ["edible", "poisonous"]

IMG_SIZE = 224
BATCH_SIZE = 32
LEARNING_RATE = 1e-4

In [4]:
# DATA GENERATOR (RESCALE ONLY)
def preprocess_with_pad(image):
    image = tf.image.resize_with_pad(image, IMG_SIZE, IMG_SIZE)
    image = preprocess_input(image)
    return image


train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad,
    rotation_range=10,
    zoom_range=0.1,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True,
    fill_mode="nearest"
)

val_test_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad
)

train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_data = val_test_gen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_data = val_test_gen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)


Found 2256 images belonging to 2 classes.
Found 282 images belonging to 2 classes.
Found 232 images belonging to 2 classes.


In [5]:
from tensorflow.keras.models import Model

# MODEL MOBILENETV3 LARGE
base_model = MobileNetV3Small(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

# Freeze feature extractor
base_model.trainable = False

In [6]:
# Build model 
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(2, activation="softmax")
])

# COMPILE MODEL
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 MobilenetV3small (Functiona  (None, 7, 7, 576)        939120    
 l)                                                              
                                                                 
 global_average_pooling2d (G  (None, 576)              0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 128)               73856     
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 2)                 258       
                                                                 
Total params: 1,013,234
Trainable params: 74,114
Non-tra

In [7]:
# ESTIMASI UKURAN MODEL (TANPA SAVE MODEL)
total_params = model.count_params()

# default dtype MobileNet = float32 (4 byte)
BYTES_PER_PARAM = 4  

estimated_size_mb = (total_params * BYTES_PER_PARAM) / (1024 * 1024)

print("\nEstimated Model Size (In-Memory)")
print(f"Total parameter      : {total_params:,}")
print(f"ukuran model: {estimated_size_mb:.2f} MB (float32)")


Estimated Model Size (In-Memory)
Total parameter      : 1,013,234
ukuran model: 3.87 MB (float32)


In [8]:
# EARLY STOPPING
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [9]:
# TRAINING + TIME MEASUREMENT
start_time = time.time()

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=25,
    callbacks=[early_stopping],
    verbose=1
)

end_time = time.time()

total_training_time = end_time - start_time
epochs_ran = len(history.history["loss"])

print(f"\nTotal Training Time : {total_training_time:.2f} seconds")
print(f"Average / Epoch    : {total_training_time / epochs_ran:.2f} seconds")
print(f"Training stopped at epoch : {epochs_ran}")

Epoch 1/25
26/71 [=========>....................] - ETA: 1:44 - loss: 0.7253 - accuracy: 0.5685

C:\Users\LENOVO\anaconda3\envs\tf212\lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


71/71 [==============================] - 252s 3s/step - loss: 0.6970 - accuracy: 0.5997 - val_loss: 0.5222 - val_accuracy: 0.7589
Epoch 2/25
71/71 [==============================] - 190s 3s/step - loss: 0.6049 - accuracy: 0.6862 - val_loss: 0.4742 - val_accuracy: 0.7766
Epoch 3/25
71/71 [==============================] - 189s 3s/step - loss: 0.5559 - accuracy: 0.7190 - val_loss: 0.4467 - val_accuracy: 0.8014
Epoch 4/25
71/71 [==============================] - 169s 2s/step - loss: 0.5256 - accuracy: 0.7420 - val_loss: 0.4305 - val_accuracy: 0.8085
Epoch 5/25
71/71 [==============================] - 117s 2s/step - loss: 0.5119 - accuracy: 0.7438 - val_loss: 0.4170 - val_accuracy: 0.8156
Epoch 6/25
71/71 [==============================] - 105s 1s/step - loss: 0.4869 - accuracy: 0.7633 - val_loss: 0.3995 - val_accuracy: 0.8262
Epoch 7/25
71/71 [==============================] - 125s 2s/step - loss: 0.4691 - accuracy: 0.7691 - val_loss: 0.3891 - val_accuracy: 0.8262
Epoch 8/25
71/71 [======

In [27]:
# Waktu Inferensi Keseluruhan Test Set
start_time = time.time()

predictions = model.predict(test_data)

end_time = time.time()

inference_time_total = end_time - start_time
num_samples = test_data.samples

print(f"Total Inference Time (Test Set): {inference_time_total:.4f} seconds")
print(f"Average Inference Time per Image: {inference_time_total / num_samples:.6f} seconds")

3/8 [==========>...................] - ETA: 5s

C:\Users\LENOVO\anaconda3\envs\tf212\lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


8/8 [==============================] - 10s 1s/step
Total Inference Time (Test Set): 11.3911 seconds
Average Inference Time per Image: 0.049100 seconds


In [28]:
# Evaluasi Model (Test Set – Tetap Sama)
test_loss, test_acc = model.evaluate(test_data)
print(f"Test Accuracy: {test_acc:.4f}")

8/8 [==============================] - 9s 1s/step - loss: 0.3231 - accuracy: 0.8793
Test Accuracy: 0.8793


In [29]:
# Ambil 1 batch
x_batch, _ = next(test_data)

# Warm-up (penting untuk CNN)
_ = model.predict(x_batch[:1])

start_time = time.time()
_ = model.predict(x_batch[:1])
end_time = time.time()

print(f"Inference Time (Single Image): {(end_time - start_time):.6f} seconds")

1/1 [==============================] - 0s 33ms/step
Inference Time (Single Image): 0.073914 seconds


In [30]:
# CONFUSION MATRIX & REPORT
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_true = test_data.classes
y_pred = np.argmax(model.predict(test_data), axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASSES))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

8/8 [==============================] - 9s 1s/step

Classification Report:
              precision    recall  f1-score   support

      edible       0.90      0.87      0.88       120
   poisonous       0.86      0.89      0.88       112

    accuracy                           0.88       232
   macro avg       0.88      0.88      0.88       232
weighted avg       0.88      0.88      0.88       232


Confusion Matrix:
[[104  16]
 [ 12 100]]


In [31]:
# simpan ke file .keras

BASE_MODEL_KERAS_PATH = r"D:\Project Jupyter\FINAL\mobilenetv3SMALL25_jamurTF2FIX.keras"

# Simpan model
model.save(BASE_MODEL_KERAS_PATH, save_format="keras")

print(f"✅ Model berhasil disimpan dalam format .keras: {BASE_MODEL_KERAS_PATH}")



✅ Model berhasil disimpan dalam format .keras: D:\Project Jupyter\FINAL\mobilenetv3SMALL25_jamurTF2FIX.keras


In [37]:
BASE_MODEL_TF_PATH = r"D:\Project Jupyter\MODEL\mobilenetv3SMALL25_jamur25COPYBARUFIXTF2PREPRO_tf"

# Simpan model
model.save(BASE_MODEL_TF_PATH, save_format="tf")

print(f"✅ Model berhasil disimpan dalam format SavedModel TensorFlow: {BASE_MODEL_TF_PATH}")


INFO:tensorflow:Assets written to: D:\Project Jupyter\MODEL\mobilenetv3SMALL25_jamur25COPYBARUFIXTF2PREPRO_tf\assets


INFO:tensorflow:Assets written to: D:\Project Jupyter\MODEL\mobilenetv3SMALL25_jamur25COPYBARUFIXTF2PREPRO_tf\assets


✅ Model berhasil disimpan dalam format SavedModel TensorFlow: D:\Project Jupyter\MODEL\mobilenetv3SMALL25_jamur25COPYBARUFIXTF2PREPRO_tf


In [38]:
import os

# Ukuran file .keras
size_keras = os.path.getsize(BASE_MODEL_KERAS_PATH) / (1024 * 1024)  # MB

# Ukuran folder SavedModel
def get_folder_size_mb(folder_path):
    total = 0
    for root, dirs, files in os.walk(folder_path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / (1024 * 1024)  # MB

size_tf = get_folder_size_mb(BASE_MODEL_TF_PATH )

print(f"Ukuran Base model (.keras)      : {size_keras:.2f} MB")
print(f"Ukuran Base model (SavedModel)  : {size_tf:.2f} MB")


Ukuran Base model (.keras)      : 4.73 MB
Ukuran Base model (SavedModel)  : 8.56 MB
